# Step 10: does the model use its rule when the moves are written out? (added after looking)

Plan and expectations committed before these answers existed: see [PLAN_more_inputs.md](../PLAN_more_inputs.md).

Test 3 again (`gemma-3-4b-it`, the same 1,296 days, 2017–2019), with two lines added below the table: the direction of the 1-, 5- and 20-day moves, and Kim's 12 indicators on day −1. Three runs: planted tester, planted none (its twin), real none.

`run_with_indicators.py` collects the answers. This notebook only reads them.

In [1]:
import sys
sys.path.insert(0, "..")  # the main project's code is one folder up

import json

import pandas as pd
from sklearn.metrics import roc_auc_score

from evaluation import mcnemar
from llm_harness import rules
from llm_harness.client import SAVED
from llm_harness.questions import days_before
from llm_harness.run_feedback import load

RUNS = [("planted", "tester"), ("planted", "none"), ("real", "none")]

def answers(dataset, kind, suffix):
    """One row per answer, with its month number (1 = the first month)."""
    months = json.loads((SAVED / f"feedback_{dataset}_{kind}{suffix}_state.json").read_text(encoding="utf-8"))["months"]
    frame = pd.DataFrame([dict(r, month=n + 1) for n, month in enumerate(months) for r in month["results"]])
    frame["right"] = (frame["answer"] == "up") == frame["up"]
    return frame.set_index(["market", "day"])

new = {(d, k): answers(d, k, "_indicators") for d, k in RUNS}
old = {(d, k): answers(d, k, "") for d, k in RUNS}  # test 3
print({key: len(a) for key, a in new.items()})

{('planted', 'tester'): 1296, ('planted', 'none'): 1296, ('real', 'none'): 1296}


## Was reading the table the problem?

Months 25–36: how often the answer matches what its own named rule gives. Test 3 got 47–55%. Answers with "none of these" are left out.

In [2]:
def late_check(a, dataset):
    markets = load(dataset)
    late = a[a["month"] > 24].reset_index()
    late["rule's answer"] = [rules.implied_answer(days_before(markets[m], pd.Timestamp(d)), rule)
                             for m, d, rule in zip(late["market"], late["day"], late["rule"])]
    named = late.dropna(subset=["rule's answer"])
    truth = named["up"].map({True: "up", False: "down"})
    return {"answers": len(named), "said up %": 100 * (named["answer"] == "up").mean(),
            "used 'reverse the 1-day move' %": 100 * (named["rule"] == "reverse the 1-day move").mean(),
            "answer agrees with its own rule %": 100 * (named["answer"] == named["rule's answer"]).mean(),
            "accuracy %": 100 * named["right"].mean(),
            "accuracy if it had obeyed its rule %": 100 * (named["rule's answer"] == truth).mean()}

pd.DataFrame({(version, d, k): late_check(runs[d, k], d) for version, runs in {"with lines": new, "test 3": old}.items()
              for d, k in RUNS}).T.round(1)

answers  said up %  \
with lines planted tester    432.0       20.1   
                   none      432.0       84.3   
           real    none      432.0       83.6   
test 3     planted tester    432.0       35.2   
                   none      432.0       95.6   
           real    none      432.0       95.4   

                           used 'reverse the 1-day move' %  \
with lines planted tester                            100.0   
                   none                               55.3   
           real    none                               47.9   
test 3     planted tester                            100.0   
                   none                                0.0   
           real    none                                0.0   

                           answer agrees with its own rule %  accuracy %  \
with lines planted tester                               35.2        44.2   
                   none                                 59.3        49.1   
           real    none                                 64.4        50.9   
test 3     planted tester                               48.4        51.4   
                   none                                 52.3        53.5   
           real    none                                 55.3        50.2   

                           accuracy if it had obeyed its rule %  
with lines planted tester                                  63.7  
                   none                                    56.5  
           real    none                                    50.5  
test 3     planted tester                                  63.7  
                   none                                    36.3  
           real    none                                    50.0

## Main results

- **Planted tester against its twin**, on the same days (McNemar). Test 3: +1.2 points.
- **Each new run against its test 3 version**, on the same days.

In [3]:
def compare(a, b):
    both = a[["right"]].join(b[["right"]].add_suffix(" other"), how="inner")
    only_a, only_b, p = mcnemar(pd.Series(True, index=both.index), both["right"], both["right other"])
    return {"days": len(both), "accuracy %": 100 * both["right"].mean(), "other %": 100 * both["right other"].mean(),
            "difference, points": 100 * (both["right"].mean() - both["right other"].mean()), "p": p}

pd.DataFrame({
    "planted tester vs its twin (with lines)": compare(new["planted", "tester"], new["planted", "none"]),
    "planted tester vs its twin (test 3)": compare(old["planted", "tester"], old["planted", "none"]),
    **{f"{d} {k}: with lines vs test 3": compare(new[d, k], old[d, k]) for d, k in RUNS},
}).T.round(3)

,days,accuracy %,other %,"difference, points",p
planted tester vs its twin (with lines),1296.0,47.840,47.840,0.000,1.000
planted tester vs its twin (test 3),1296.0,50.463,49.228,1.235,0.591
planted tester: with lines vs test 3,1296.0,47.840,50.463,-2.623,0.138
planted none: with lines vs test 3,1296.0,47.840,49.228,-1.389,0.288
real none: with lines vs test 3,1296.0,50.694,50.926,-0.231,0.900


## Any skill on real data?

AUC of the real none run, with the lines and without (test 3). The step 9 models with set A, on the same days, are below.

In [4]:
pd.Series({"with lines": roc_auc_score(new["real", "none"]["up"], new["real", "none"]["p_up"]),
           "test 3": roc_auc_score(old["real", "none"]["up"], old["real", "none"]["p_up"])}, name="AUC").round(3).to_frame()

,AUC
with lines,0.501
test 3,0.511


In [5]:
import os
os.chdir("..")  # step 9's saved runs are found from the project folder
from more_inputs import MODELS, SAVED as STEP9
a = new["real", "none"].reset_index()
a["day"] = pd.to_datetime(a["day"])
rows = {}
for model in MODELS:
    ml = pd.concat([pd.read_pickle(STEP9 / f"A_close_to_close_{model}_{m}.pkl".replace(" ", "_")).assign(market=m)
                    for m in a["market"].unique()]).reset_index(names="day")
    both = a.merge(ml, on=["market", "day"], suffixes=("", " ml"))
    rows[model] = {"days": len(both), "AUC": roc_auc_score(both["up ml"], both["p_up ml"])}
os.chdir("llm_harness")
pd.DataFrame(rows).T.round(3)

,days,AUC
LDA,1295.0,0.502
Logistic regression,1295.0,0.496
KNN,1295.0,0.513
SVM,1295.0,0.507
Random forest,1295.0,0.512
AdaBoost,1295.0,0.515
Neural network,1295.0,0.507


## Added after looking: does it copy the direction it is shown?

In the planted tester run its answers agreed with its own rule *less* than a coin flip. Months 25–36: how often the answer simply equals the direction of each move (as "follow the move" would give).

In [6]:
def copies(a, dataset):
    markets = load(dataset)
    late = a[a["month"] > 24].reset_index()
    tables = [days_before(markets[m], pd.Timestamp(d)) for m, d in zip(late["market"], late["day"])]
    return {f"answer = direction of the {w} %": 100 * pd.Series([rules.implied_answer(t, f"follow the {w}") for t in tables]).eq(late["answer"]).mean()
            for w in rules.WINDOWS}

pd.DataFrame({(version, d, k): copies(runs[d, k], d) for version, runs in {"with lines": new, "test 3": old}.items()
              for d, k in RUNS}).T.round(1)

answer = direction of the 1-day move %  \
with lines planted tester                                    64.8   
                   none                                      60.9   
           real    none                                      61.6   
test 3     planted tester                                    51.6   
                   none                                      52.3   
           real    none                                      55.3   

                           answer = direction of the 5-day move %  \
with lines planted tester                                    46.8   
                   none                                      56.2   
           real    none                                      61.6   
test 3     planted tester                                    44.7   
                   none                                      50.0   
           real    none                                      57.2   

                           answer = direction of the 20-day move %  
with lines planted tester                                     38.2  
                   none                                       43.5  
           real    none                                       53.7  
test 3     planted tester                                     26.9  
                   none                                       52.1  
           real    none                                       58.1

## Conclusions

**Writing out the moves didn't help. It made the model copy the direction it was shown, whatever rule it named.**

| Expectation from the plan | What happened |
|---|---|
| Answer agrees with its own rule 65–80% (test 3: 47–55%) | **No.** Planted tester: **35.2%**, *lower* than test 3's 48.4% and below a coin flip. The two none runs: 59.3% and 64.4% |
| Planted tester beats its twin by 3 to 8 points (test 3: +1.2) | **No.** Exactly 0.0 points (47.8% each, p = 1.0) |
| The twin falls toward 40%, by obeying "follow the 1-day move" | **No.** 47.8% overall (test 3: 49.2%). With the moves in front of it, the twin named "reverse the 1-day move" in 55% of late answers instead |
| Real data: AUC 0.48–0.53, no better than set A's models | **As expected.** AUC 0.501 (test 3: 0.511). Step 9's models with set A on the same 1,295 days: 0.496–0.515 |

**Why (added after looking):** with the moves written out, its answer equals the direction of the 1-day move 61–65% of the time, against 52–55% in test 3. In the planted tester run it named "reverse the 1-day move" in every late answer, but answered the way the 1-day move went, the opposite of that rule. Its answers agreed with "reverse" 35% of the time, so with "follow" about 65%. On planted data, where reversing is right, that cost it: 44.2% in months 25–36.

**What this says about tests 2 and 3:** the problem wasn't only reading the numbers. Given the direction as a word, the model does use it, but it copies the word instead of applying its rule. Naming a rule and acting on it are separate things for this model, and the second doesn't follow from the first.

**In short:** a 4-billion-parameter model can pick the right rule from a scorecard and be handed exactly what that rule needs, and still not apply it. Turning "down" into "up" is the step it fails.